In [12]:
# Install Python's R pip
!pip install rpy2 numpy pandas scipy matplotlib
!pip install pyreadr

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------------------------- ---- 2.1/2.4 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.8 MB/s  0:00:00


In [2]:
import os
os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.3"
os.environ["PATH"]   = r"C:\Program Files\R\R-4.5.3\bin\x64;" + os.environ["PATH"]

In [1]:
# Environment Setting
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyreadr
import rpy2.robjects as ro

In [3]:
r = ro.r

In [4]:
# Read data
RDATA_PATH = "blm.RData"
result = pyreadr.read_r(RDATA_PATH)
old_clean = result["old_clean"]

In [5]:
dates = pd.to_datetime(old_clean.iloc[:,0].astype(str), format="%Y%m%d")
x0 = old_clean.iloc[:,1:].values.T.astype(float)
p, T_total = x0.shape

n = 252
kern_bw = int(np.floor(4*(n/np.log(n))**(1/3)))
print(f"Data: p={p}, T={T_total}, kern_bw={kern_bw}")

Data: p=46, T=3269, kern_bw=14


In [6]:
# Regime split
date_int = old_clean.iloc[:, 0].astype(int).values  #yyyymmdd

tt_calm = np.where((date_int >= 20040101) & (date_int <= 20061231))[0]
tt_crisis = np.where((date_int >= 20070101) & (date_int <= 20091231))[0]

tt_calm = tt_calm[tt_calm > n]
tt_crisis = tt_crisis[tt_crisis > n]

print(f"Calm : {len(tt_calm)} days")
print(f"Crisis: {len(tt_crisis)} days")

Calm : 755 days
Crisis: 756 days


In [7]:
# Assumption Tests

def check_assumptions(x_sub: np.ndarray, label: str):

    """x_sub : (p, T_sub)"""
    p_, T_ = x_sub.shape
    print("=" * 40)
    print(f"Period: {label}")
    print(f"Dimensions: p={p_}, T={T_}")
    print("=" * 40)

    # A1: Stationarity - ADF test
    adf_pvals = np.array([adfuller(x_sub[i, :])[1] for i in range(p_)])
    print(f"\n A1: Stationarity (ADF Test)")
    print(f"  % stationary (p<0.05) : {np.mean(adf_pvals < 0.05)*100:.1f}%")
    print(f"  Min p-value           : {adf_pvals.min():.4f}")
    print(f"  Max p-value           : {adf_pvals.max():.4f}")

    # A3: Heavy Tails

    kurt = np.array([stats.kurtosis(x_sub[i, :], fisher = False) for i in range(p_)])
    jb_pvals = np.array([stats.jarque_bera(x_sub[i, :])[1] for i in range(p_)])
    ag_pvals = np.array([stats.kurtosistest(x_sub[i, :])[1] for i in range(p_)])

    print(f"\n A3: Heavy Tails")
    print(f"  Mean kurtosis                    : {np.mean(kurt):.3f}")
    print(f"  % JB reject normality (p<0.05)   : {np.mean(jb_pvals < 0.05)*100:.1f}%")
    print(f"  % AG reject kurtosis=3 (p<0.05)  : {np.mean(ag_pvals < 0.05)*100:.1f}%")

In [8]:
check_assumptions(x0[:, tt_calm],   "Calm Period (2004-2006)")
check_assumptions(x0[:, tt_crisis], "Crisis Period (2007-2009)")

Period: Calm Period (2004-2006)
Dimensions: p=46, T=755

 A1: Stationarity (ADF Test)
  % stationary (p<0.05) : 100.0%
  Min p-value           : 0.0000
  Max p-value           : 0.0185

 A3: Heavy Tails
  Mean kurtosis                    : 3.392
  % JB reject normality (p<0.05)   : 63.0%
  % AG reject kurtosis=3 (p<0.05)  : 34.8%
Period: Crisis Period (2007-2009)
Dimensions: p=46, T=756

 A1: Stationarity (ADF Test)
  % stationary (p<0.05) : 30.4%
  Min p-value           : 0.0041
  Max p-value           : 0.3631

 A3: Heavy Tails
  Mean kurtosis                    : 2.838
  % JB reject normality (p<0.05)   : 76.1%
  % AG reject kurtosis=3 (p<0.05)  : 41.3%


In [9]:
# FNETS

def run_fnets_r(x0_np: np.ndarray, tt_ind: np.ndarray,
                n_win: int, kern_bw_val: int, tag: str):
    
    T_ = x0_np.shape[1]
    p_ = x0_np.shape[0]
 
    x0_flat  = x0_np.flatten(order='F')
    x0_str   = ",".join(map(str, x0_flat))
    tt_str   = ",".join(map(str, (tt_ind + 1).tolist()))  # R 1-based index
 
    r(f'''
library(glmnet)
library(fnets)
 
x0_r     <- matrix(c({x0_str}), nrow={p_}, ncol={T_})
p_r      <- {p_}
n_r      <- {n_win}
kern_bw  <- {kern_bw_val}
tt_ind_r <- c({tt_str})
 
T_total      <- {T_}
edge_density <- rep(NA, T_total)
s_in_vec     <- rep(NA, T_total)
q_vec        <- rep(NA, T_total)
fle_avg      <- rep(NA, T_total)
fle_max      <- rep(NA, T_total)
fl_store     <- matrix(NA, nrow=p_r, ncol=T_total)
 
cat("Starting FNETS [{tag}], n={n_win}, kern_bw={kern_bw_val}\\n")
 
for (tt in tt_ind_r) {{
 
  yl     <- matrix(0, nrow=p_r, ncol=1)
  int    <- max(1, tt - n_r):(tt - 1)
  x      <- x0_r[, int]
  x_true <- x0_r[, tt]
  mean_x <- rowMeans(x)
  xx     <- x - mean_x
 
  # Step 1: Dynamic PCA
  dpca <- fnets:::dyn.pca(xx, q=NULL, ic.op=5, kern.bw=kern_bw)
  q    <- dpca$q
  if (is.na(q) || q == 0) {{
    dpca <- fnets:::dyn.pca(xx, q=1, ic.op=5, kern.bw=kern_bw)
    q    <- dpca$q
  }}
  acv  <- dpca$acv
  spec <- dpca$spec
 
  cve <- fnets:::common.irf.estimation(xx, Gamma_c=acv$Gamma_c, q=q,
             factor.var.order=NULL, max.var.order=NULL,
             trunc.lags=20, n.perm=30)
 
  obj <- list(acv=acv, spec=spec, loadings=cve$irf.est,
              factors=cve$u.est, q=q, mean.x=mean_x, kern.bw=kern_bw)
  attr(obj, "factor") <- "unrestricted"
 
  cpre_static <- fnets:::common.predict(obj, x, n.ahead=1, r="ic", fc.restricted=TRUE)
 
  # Step 2+3: Yule-Walker LASSO
  lam_max  <- max(abs(xx %*% t(xx) / n_r))
  lam_path <- round(exp(seq(log(lam_max), log(lam_max * 1e-4), length.out=20)), 10)
  lambda   <- lam_path[9]
 
  mg  <- fnets:::make.gg(acv$Gamma_i, 1)
  ive <- fnets:::var.lasso(mg$GG, mg$gg, lambda)
 
  obj$idio.var <- ive
  ipre    <- fnets:::idio.predict(obj, x, cpre_static, n.ahead=1)
  yl[,1]  <- mean_x + cpre_static$fc + ipre$fc
 
  # Sparsity
  G                <- ive$beta != 0
  edge_density[tt] <- sum(G) / (p_r * p_r)
  s_in_vec[tt]     <- max(colSums(G))
  q_vec[tt]        <- q
 
  # Forecast errors
  fl_store[, tt] <- yl
  fle_avg[tt]    <- sum((x_true - yl)^2) / sum(x_true^2)
  fle_max[tt]    <- max(abs(x_true - yl)) / max(abs(x_true))
}}
cat("FNETS [{tag}] done.\\n")
''')
 
    fle_avg_py      = np.array(r("fle_avg"))
    fle_max_py      = np.array(r("fle_max"))
    edge_density_py = np.array(r("edge_density"))
    s_in_vec_py     = np.array(r("s_in_vec"))
    q_vec_py        = np.array(r("q_vec"))
 
    return {
        "fle_avg":      fle_avg_py,
        "fle_max":      fle_max_py,
        "edge_density": edge_density_py,
        "s_in_vec":     s_in_vec_py,
        "q_vec":        q_vec_py,
    }
 
 
tt_ind_all = np.concatenate([tt_calm, tt_crisis])


In [10]:
print("\n=== Running FNETS n=252 ===")
ls = run_fnets_r(x0, tt_ind_all, n_win=252, kern_bw_val=kern_bw, tag="n=252")
 
print("\n=== Running FNETS n=126 (Robustness) ===")
n126       = 126
kern_bw126 = int(np.floor(4 * (n126 / np.log(n126)) ** (1/3)))
ls126 = run_fnets_r(x0, tt_ind_all, n_win=n126, kern_bw_val=kern_bw126, tag="n=126")


=== Running FNETS n=252 ===


R callback write-console: Loading required package: Matrix
  
R callback write-console: Loaded glmnet 4.1-10
  


Starting FNETS [n=252], n=252, kern_bw=14
FNETS [n=252] done.

=== Running FNETS n=126 (Robustness) ===
Starting FNETS [n=126], n=126, kern_bw=11
FNETS [n=126] done.


In [11]:
# Results Summary

def print_results(ls_dict: dict, tt_c: np.ndarray, tt_cr: np.ndarray, tag: str):
    for period, tt_p in [("calm", tt_c), ("crisis", tt_cr)]:
        avg = ls_dict["fle_avg"][tt_p]
        mx  = ls_dict["fle_max"][tt_p]
        print(f"\n[{tag}] {period}")
        print(f"  avg  mean={np.nanmean(avg):.4f}  median={np.nanmedian(avg):.4f}  sd={np.nanstd(avg):.4f}")
        print(f"  max  mean={np.nanmean(mx):.4f}  median={np.nanmedian(mx):.4f}  sd={np.nanstd(mx):.4f}")
 
    for period, tt_p in [("Calm", tt_c), ("Crisis", tt_cr)]:
        print(f"\n  === [{tag}] {period} sparsity ===")
        print(f"  Edge density mean : {np.nanmean(ls_dict['edge_density'][tt_p]):.4f}")
        print(f"  s_in mean         : {np.nanmean(ls_dict['s_in_vec'][tt_p]):.2f}")
        print(f"  q mean            : {np.nanmean(ls_dict['q_vec'][tt_p]):.2f}")
 
print_results(ls,    tt_calm, tt_crisis, "n=252")
print_results(ls126, tt_calm, tt_crisis, "n=126")


[n=252] calm
  avg  mean=0.5884  median=0.4006  sd=0.6062
  max  mean=0.8526  median=0.7858  sd=0.2382

[n=252] crisis
  avg  mean=0.5529  median=0.3827  sd=0.5882
  max  mean=0.7443  median=0.7107  sd=0.2625

  === [n=252] Calm sparsity ===
  Edge density mean : 0.1008
  s_in mean         : 11.06
  q mean            : 1.15

  === [n=252] Crisis sparsity ===
  Edge density mean : 0.1316
  s_in mean         : 17.34
  q mean            : 1.60

[n=126] calm
  avg  mean=0.5944  median=0.3904  sd=0.6394
  max  mean=0.8552  median=0.7864  sd=0.2490

[n=126] crisis
  avg  mean=0.5620  median=0.3806  sd=0.6121
  max  mean=0.7537  median=0.7117  sd=0.2705

  === [n=126] Calm sparsity ===
  Edge density mean : 0.1636
  s_in mean         : 15.10
  q mean            : 1.01

  === [n=126] Crisis sparsity ===
  Edge density mean : 0.2528
  s_in mean         : 23.75
  q mean            : 1.14


In [12]:
# t-test

def ttest(a, b, label):
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)
    print(f"{label}")
    print(f"  t={t_stat:.4f},  p={p_val:.4f},  n_calm={len(a)},  n_crisis={len(b)}")
 
print("\n=== t-tests ===")
ttest(ls["fle_avg"][tt_calm],    ls["fle_avg"][tt_crisis],    "n=252  FE_avg")
ttest(ls["fle_max"][tt_calm],    ls["fle_max"][tt_crisis],    "n=252  FE_max")
ttest(ls126["fle_avg"][tt_calm], ls126["fle_avg"][tt_crisis], "n=126  FE_avg")
ttest(ls126["fle_max"][tt_calm], ls126["fle_max"][tt_crisis], "n=126  FE_max")


=== t-tests ===
n=252  FE_avg
  t=1.1567,  p=0.2476,  n_calm=755,  n_crisis=756
n=252  FE_max
  t=8.3941,  p=0.0000,  n_calm=755,  n_crisis=756
n=126  FE_avg
  t=1.0075,  p=0.3139,  n_calm=755,  n_crisis=756
n=126  FE_max
  t=7.5836,  p=0.0000,  n_calm=755,  n_crisis=756


In [13]:
# Plots

CALM_C   = "#2166ac"
CRISIS_C = "#d6604d"
CALM_F   = "#92c5de"
CRISIS_F = "#f4a582"
 
all_tt      = np.concatenate([tt_calm, tt_crisis])
all_dates   = dates.iloc[all_tt].values
is_calm     = np.array([True]*len(tt_calm) + [False]*len(tt_crisis))
 
# ── Fig 0: Cross-sectional mean log-volatility ──
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(dates.iloc[tt_calm].values,   x0[:, tt_calm].mean(axis=0),
        color=CALM_C,   lw=0.5, alpha=0.8, label="Calm (2004-2006)")
ax.plot(dates.iloc[tt_crisis].values, x0[:, tt_crisis].mean(axis=0),
        color=CRISIS_C, lw=0.5, alpha=0.8, label="Crisis (2007-2009)")
ax.axhline(x0[:, tt_calm].mean(),   color=CALM_C,   ls="--", lw=0.7)
ax.axhline(x0[:, tt_crisis].mean(), color=CRISIS_C, ls="--", lw=0.7)
ax.set_title("Figure 3: Cross-Sectional Mean Log-Volatility", fontsize=11)
ax.set_ylabel("Mean log-volatility");  ax.legend(fontsize=8)
fig.tight_layout();  fig.savefig("fig0_mean_logvol.pdf");  plt.close()
 
# ── Fig 1: FE_max time series ──
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(all_dates[is_calm],  ls["fle_max"][all_tt][is_calm],
        color=CALM_C,   lw=0.4, alpha=0.8, label="Calm (2004-2006)")
ax.plot(all_dates[~is_calm], ls["fle_max"][all_tt][~is_calm],
        color=CRISIS_C, lw=0.4, alpha=0.8, label="Crisis (2007-2009)")
ax.axvline(pd.Timestamp("2007-01-01"), color="grey", ls="--", lw=0.6)
ax.set_title("Figure 2: Max Forecast Error Over Time", fontsize=11)
ax.set_ylabel("FE_max");  ax.legend(fontsize=8)
fig.tight_layout();  fig.savefig("fig1_femax_timeseries.pdf");  plt.close()
 
# ── Fig 2: Edge density time series ──
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(all_dates[is_calm],  ls["edge_density"][all_tt][is_calm],
        color=CALM_C,   lw=0.4, alpha=0.8, label="Calm (2004-2006)")
ax.plot(all_dates[~is_calm], ls["edge_density"][all_tt][~is_calm],
        color=CRISIS_C, lw=0.4, alpha=0.8, label="Crisis (2007-2009)")
ax.axvline(pd.Timestamp("2007-01-01"), color="grey", ls="--", lw=0.6)
ax.set_title("Figure 1: Network Edge Density Over Time", fontsize=11)
ax.set_ylabel("Edge Density");  ax.legend(fontsize=8)
fig.tight_layout();  fig.savefig("fig2_edge_density.pdf");  plt.close()
 
# ── Fig 3 & 4: Boxplots ──
def save_boxplot(calm_252, crisis_252, calm_126, crisis_126, ylabel, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)
    for ax, (c_vals, cr_vals, tag) in zip(axes, [
        (calm_252,  crisis_252,  "n = 252"),
        (calm_126,  crisis_126,  "n = 126"),
    ]):
        c_vals  = c_vals[~np.isnan(c_vals)]
        cr_vals = cr_vals[~np.isnan(cr_vals)]
        bp = ax.boxplot([c_vals, cr_vals], patch_artist=True,
                        flierprops=dict(markersize=1, alpha=0.3))
        bp["boxes"][0].set_facecolor(CALM_F)
        bp["boxes"][1].set_facecolor(CRISIS_F)
        ax.set_xticklabels(["Calm", "Crisis"])
        ax.set_title(tag, fontsize=10)
    axes[0].set_ylabel(ylabel)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout();  fig.savefig(fname);  plt.close()
 
save_boxplot(
    ls["fle_avg"][tt_calm],    ls["fle_avg"][tt_crisis],
    ls126["fle_avg"][tt_calm], ls126["fle_avg"][tt_crisis],
    "FE_avg", "Figure 4: Average Forecast Error by Period and Window Size",
    "fig3_boxplot_avg.pdf"
)
save_boxplot(
    ls["fle_max"][tt_calm],    ls["fle_max"][tt_crisis],
    ls126["fle_max"][tt_calm], ls126["fle_max"][tt_crisis],
    "FE_max", "Figure 5: Max Forecast Error by Period and Window Size",
    "fig4_boxplot_max.pdf"
)
 
print("\nAll figures saved. Done!")



All figures saved. Done!
